# 60 — Prospective Protocol Freeze

Congela **antes de observar resultados futuros** el protocolo de M7: fechas, target, métricas, thresholds, modelo/calibrador, scoring contract y reglas de gobernanza.

Este notebook es el preregistro técnico de la evaluación prospectiva. TEST 2025 y OOT 2026 permanecen consumidos.


In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.prospective import load_prospective_spec
from argos.config.production import load_production_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
M7_SPEC = load_prospective_spec(paths.configs / "prospective.toml")
M6_SPEC = load_production_spec(paths.configs / "production.toml")
(paths.reports / "prospective").mkdir(parents=True, exist_ok=True)
(paths.root / "artifacts" / "prospective").mkdir(parents=True, exist_ok=True)

m6_gate = paths.docs / "decisions" / "M6_GATE.md"
assert m6_gate.exists() and "M6 ✅ CLOSED" in m6_gate.read_text(encoding="utf-8"), "M6 must be CLOSED before M7."
print("M7 schema:", M7_SPEC.schema_version)
print("Protocol freeze:", M7_SPEC.core.protocol_freeze_date)
print("Prospective start:", M7_SPEC.core.prospective_start)


In [ ]:
from argos.prospective.protocol import PROTOCOL_IMPLEMENTATION, freeze_m7_protocol

print("implementation:", PROTOCOL_IMPLEMENTATION)
result = freeze_m7_protocol(
    root=paths.root,
    prospective_spec=M7_SPEC,
    production_spec=M6_SPEC,
    reports_root=paths.reports,
    docs_root=paths.docs,
)
print("protocol status:", result.status)
result.validation


In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
print("60 ✅ PASS" if result.ready_for_prospective_collection else "60 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Protocol manifest:", result.manifest_path)
assert result.ready_for_prospective_collection
